In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/LZ46LTL6O2.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/13H1VTAZ42.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/G0DWU2GK1L.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/CVPHC0DT1P.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/90XJFQDL7G.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/3V8LZ56MOW.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/Y9EUV6TNP2.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/K76P42J0GK.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/ZSS7THB1UM.jpg
/kaggle/input/comp

In [2]:
import torch.nn.functional as F, glob, os, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
 
ROOT = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
N_TRAIN, EPOCHS, BS, MIN_AREA, THR = 400, 2, 32, 50, 0.5
SCALE = 4  # work at 1/4 resolution so CPU is fast; masks are upsampled back at the end
torch.set_num_threads(os.cpu_count())
dev = "cpu"  # CPU only
 
paths = {os.path.basename(p)[:-4]: p for p in glob.glob(f"{ROOT}/**/*.jpg", recursive=True)}
train = pd.read_csv(f"{ROOT}/train.csv", keep_default_na=False, dtype=str)
sub = pd.read_csv(f"{ROOT}/sample_submission.csv", keep_default_na=False, dtype=str)
 
W, H = Image.open(paths[train.image_id.iloc[0]]).size
print("image size (H, W):", H, W)  # expected 128 x 800
 
def rle_decode(s):
    m = np.zeros(H * W, np.uint8)
    a = s.split()
    for st, ln in zip(map(int, a[0::2]), map(int, a[1::2])):
        m[st - 1:st - 1 + ln] = 1          # 1-indexed; change to `st:st+ln` if LB suggests 0-indexed
    return m.reshape((W, H)).T             # column-major
 
def rle_encode(m):
    p = np.concatenate([[0], m.T.flatten(), [0]])   # column-major
    r = np.where(p[1:] != p[:-1])[0] + 1
    r[1::2] -= r[::2]
    return " ".join(map(str, r)) if len(r) else " "
 
class DS(Dataset):
    def __init__(self, ids, masks=None): self.ids, self.masks = ids, masks
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x = np.asarray(Image.open(paths[self.ids[i]]).convert("RGB").resize((W // SCALE, H // SCALE)), np.float32) / 255.
        x = torch.from_numpy(x).permute(2, 0, 1)
        if self.masks is None: return x
        y = np.stack([rle_decode(self.masks[self.ids[i]].get(d, " "))[::SCALE, ::SCALE] for d in DEFECTS]).astype(np.float32)
        return x, torch.from_numpy(y)
 
def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
class TinyUNet(nn.Module):
    def __init__(s):
        super().__init__()
        s.e1, s.e2, s.e3 = block(3, 8), block(8, 16), block(16, 32)
        s.pool, s.up = nn.MaxPool2d(2), nn.Upsample(scale_factor=2)
        s.d2, s.d1 = block(48, 16), block(24, 8)
        s.out = nn.Conv2d(8, 4, 1)
    def forward(s, x):
        a = s.e1(x); b = s.e2(s.pool(a)); c = s.e3(s.pool(b))
        d = s.d2(torch.cat([s.up(c), b], 1)); d = s.d1(torch.cat([s.up(d), a], 1))
        return s.out(d)
 
# ---- train ----
masks = {i: dict(zip(g.defect_type, g.mask_rle)) for i, g in train.groupby("image_id")}
ids = list(masks)[:N_TRAIN]
dl = DataLoader(DS(ids, masks), batch_size=BS, shuffle=True, num_workers=4)
model = TinyUNet().to(dev)
opt = torch.optim.Adam(model.parameters(), 1e-3)
lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(5., device=dev))
for ep in range(EPOCHS):
    model.train(); tot = 0
    for x, y in dl:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(); l = lossf(model(x), y); l.backward(); opt.step(); tot += l.item()
    print(f"epoch {ep+1}: loss {tot/len(dl):.4f}")
 
# ---- predict ----
test_ids = list(dict.fromkeys(sub.image_id))
model.eval(); pred = {}
with torch.no_grad():
    for k in range(0, len(test_ids), 32):
        b = test_ids[k:k + 32]
        x = torch.stack([DS(b)[j] for j in range(len(b))]).to(dev)
        o = F.interpolate(model(x), size=(H, W), mode="bilinear", align_corners=False)
        p = (torch.sigmoid(o) > THR).cpu().numpy().astype(np.uint8)
        for j, i in enumerate(b):
            for c, d in enumerate(DEFECTS):
                m = p[j, c]
                pred[(i, d)] = rle_encode(m) if m.sum() >= MIN_AREA else " "   # drop tiny blobs -> fewer FPs
 
sub["mask_rle"] = [pred[(i, d)] for i, d in zip(sub.image_id, sub.defect_type)]
sub.to_csv("submission.csv", index=False)
print(sub.mask_rle.str.strip().ne("").mean(), "fraction non-empty"); sub.head()
 

image size (H, W): 128 800
epoch 1: loss 0.6236
epoch 2: loss 0.5803
0.20958333333333334 fraction non-empty


,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,9451 1 9457 3 9578 10 9705 10 9833 10 9960 11 ...
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
